# Intercoder Reliability Confidence Interval (R2-3.3)

Bootstrap 95% confidence interval for Cohen's kappa between the two human coders (Section 3.1), with the agreement of the LLM-assisted labels against the human consensus. B = 10,000 resamples of accounts, seed 42.

The coder sheets (`coding_coder1.csv`, `coding_coder2.csv`) and the LLM label file (`ai_labels_HIDDEN.csv`) contain account identifiers and are not published. The notebook is provided with its outputs as a record of the reported statistics.

In [1]:
import numpy as np, pandas as pd
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score
from statsmodels.stats.proportion import proportion_confint
B, SEED = 10_000, 42
c1 = pd.read_csv('coding_coder1.csv', sep=';')
c2 = pd.read_csv('coding_coder2.csv', sep=';')
ai = pd.read_csv('ai_labels_HIDDEN.csv')
m = c1.merge(c2, on='username', suffixes=('_1', '_2')).merge(ai, on='username')
x1, x2 = m['coder_label_1'].to_numpy(), m['coder_label_2'].to_numpy()

kappa = cohen_kappa_score(x1, x2)
rng = np.random.default_rng(SEED)
boot = [cohen_kappa_score(x1[i], x2[i]) for i in (rng.integers(0, len(m), len(m)) for _ in range(B))]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f'accounts = {len(m)}, raw agreement = {100*(x1 == x2).mean():.1f}%')
print(f"Cohen's kappa = {kappa:.3f}, 95% CI [{lo:.3f}, {hi:.3f}]")

accounts = 437, raw agreement = 93.1%
Cohen's kappa = 0.890, 95% CI [0.849, 0.926]


In [2]:
cons = m[m['coder_label_1'] == m['coder_label_2']]
hit = cons['account_type'] == cons['coder_label_1']
lo, hi = proportion_confint(hit.sum(), len(cons), method='wilson')
nop = cons[~cons['has_profile_1'].astype(str).str.upper().eq('TRUE')]
print(f'LLM labels vs human consensus: {hit.sum()}/{len(cons)} = {100*hit.mean():.1f}%, 95% Wilson CI [{100*lo:.1f}, {100*hi:.1f}]')
print(f'accounts without profile data: {100*(nop["account_type"] == nop["coder_label_1"]).mean():.1f}% (n = {len(nop)})')
labs = ['elite/political', 'grassroot', 'media/news']
prec = precision_score(cons['coder_label_1'], cons['account_type'], labels=labs, average=None)
rec = recall_score(cons['coder_label_1'], cons['account_type'], labels=labs, average=None)
for l, p, r in zip(labs, prec, rec): print(f'{l:<16} precision {p:.2f}  recall {r:.2f}')

LLM labels vs human consensus: 374/407 = 91.9%, 95% Wilson CI [88.8, 94.2]
accounts without profile data: 87.6% (n = 226)
elite/political  precision 0.98  recall 0.91
grassroot        precision 0.95  recall 0.89
media/news       precision 0.80  recall 1.00
